# 03 · Model training & evaluation

Launches the full training pipeline — equivalent to running:

```bash
poetry run milk -c ppl/utils/experiment_configs/run_config.yaml
```

The data uses a **predefined split** (`split` column: 0=train, 1=val, 2=test), so
there is no cross-validation:

**Stage 1** trains on the train split, selects the best checkpoint on the
validation split, and evaluates it on the test split.
**Stage 2** (only if `trainer.stage_2_launch = true`) refits on train+val and
evaluates on the held-out test set. Metrics and artifacts are logged to MLflow
and to the experiment results directory.

> ⚠️ This is the expensive step. With the default config it trains up to
> `max_epochs` on the configured `device`. Lower `max_epochs` in the YAML for a
> quick smoke test.

In [ ]:
# --- Bootstrap: make the notebook run from anywhere ---
import os, sys, logging
from pathlib import Path

# Locate the project root (folder that contains the `ppl` package).
here = Path.cwd()
PROJECT_ROOT = next(
    (p for p in [here, *here.parents] if (p / 'ppl' / '__init__.py').exists()),
    None,
)
if PROJECT_ROOT is None:
    # Fallback: this notebook lives in <root>/notebooks/
    PROJECT_ROOT = Path('__file__' in globals() and __file__ or '.').resolve().parent.parent

os.chdir(PROJECT_ROOT)                       # pipeline writes outputs relative to cwd
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

logging.basicConfig(level=logging.INFO, format='%(levelname)s %(name)s: %(message)s')
print('Project root:', PROJECT_ROOT)

In [ ]:
# Path to the experiment YAML. Edit this to point at a different config.
CONFIG_PATH = PROJECT_ROOT / 'ppl/utils/experiment_configs/run_config.yaml'
assert CONFIG_PATH.exists(), f'Config not found: {CONFIG_PATH}'
print('Using config:', CONFIG_PATH.relative_to(PROJECT_ROOT))

## Reproducibility

The CLI seeds everything before running; we do the same so notebook runs match
`poetry run milk`.

In [ ]:
from ppl.utils.reproducibility import set_deterministic, check_determinism
from ppl.utils.cli.pipeline_setup_utils import DEFAULT_SEED

set_deterministic(seed=DEFAULT_SEED)
check_determinism()
print('deterministic setup done, seed =', DEFAULT_SEED)

## Option A — run the whole pipeline (recommended)

`build_pipeline_from_config` builds a `PipelineOrchestrator` (config →
components), and `.run()` executes Stage 1 + Stage 2 with automatic cleanup.
This is the exact code path the CLI uses.

In [ ]:
from ppl.utils import build_pipeline_from_config

pipeline = build_pipeline_from_config(yaml_path=CONFIG_PATH)
pipeline.run()
print('Pipeline finished.')

## Option B — drive the stages manually

Prefer to run Stage 1 and Stage 2 separately (e.g. to inspect validation metrics
before the final refit)? Build the components yourself and call each stage. Run
**either** Option A **or** Option B, not both.

In [ ]:
# from ppl.utils.modelling_configs.pipeline_config import PipelineConfig
# from ppl.utils.pipeline.pipeline_initialization import initialize_pipeline_components
#
# cfg = PipelineConfig.from_yaml(CONFIG_PATH)
# config_mgr, resource_mgr, factory, task = initialize_pipeline_components(cfg)
#
# # Stage 1: train on split=0, validate on split=1, test the best checkpoint on split=2
# val_metrics = factory.split_trainer.run()
# print('validation metrics:', val_metrics)
#
# # Stage 2 (optional): final refit on train+val, evaluate on test
# if getattr(cfg.trainer, 'stage_2_launch', False):
#     factory.model_evaluator.final_fit_test()

## Where results land

- **MLflow**: launch the UI from the project root to browse runs/metrics:
  ```bash
  mlflow ui --backend-store-uri exp_log/mlflow_logs
  ```
- **Results directory**: predictions (`train_fit.csv`, `val.csv`, `test.csv`),
  `res.txt` summary, attention-weight plots and true-vs-pred figures are written
  under the experiment results folder.

In [ ]:
from ppl.utils.pipeline.results_directory import create_results_directory
from ppl.utils.pipeline.config_manager import PipelineConfigManager
from ppl.utils.modelling_configs.pipeline_config import PipelineConfig

exp_name = PipelineConfigManager(PipelineConfig.from_yaml(CONFIG_PATH)).trainer_cfg.experiment_name
results_dir = create_results_directory(exp_name)
print('Results dir:', results_dir)
for p in sorted(results_dir.rglob('*'))[:40]:
    print(' ', p.relative_to(results_dir))